In [ ]:
import os
from pathlib import Path
from pyspark.sql.functions import col, regexp_replace, substring_index,input_file_name, current_timestamp
#Parameters assignment from task
banfield_catalog = dbutils.widgets.get("banfield_catalog")
bf_vwmvhcore_catalog = dbutils.widgets.get("bf_vwmvhcore_catalog")
bf_vwedh_catalog = dbutils.widgets.get("bf_vwedh_catalog")
bf_vwvoyager_catalog = dbutils.widgets.get("bf_vwvoyager_catalog")

df = (
   spark.read
   .option("header", "true")
   .option("delimiter", ",")
   .option("quote", '"')
   .option("escape", '"')
   .option("multiLine", "true")
   .option("inferSchema", "false")
   .csv(f"/Volumes/{banfield_catalog}/bfdw_iron/inbound/Voyager/current_hosp*.csv")
   .withColumn("SOURCE_FILE", substring_index(col("_metadata.file_path"), "/", -1))
   .withColumn("INGESTION_TS", current_timestamp())
)

df = df.toDF(*[c.strip('"') for c in df.columns])
# Strip surrounding double quotes from string columns
for c in df.columns:
    if dict(df.dtypes)[c] == 'string':
        df = df.withColumn(c, regexp_replace(col(c), '^"|"$', ''))
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{banfield_catalog}.bfdw_iron.cmn_tbcmdcurrenthosp")

file_path = f"/Volumes/{banfield_catalog}/bfdw_iron/inbound/Voyager/"
files_to_delete = [str(f.path) for f in dbutils.fs.ls(file_path) if f.name.startswith("current_hosp_") and f.name.endswith(".csv")]

for file_path in files_to_delete:
    dbutils.fs.rm(file_path, True)